# Trợ lý Thị giác AI — Kaggle Dual T4

Notebook chạy hệ thống từ mã nguồn private đã được Kaggle Dataset tự giải nén trong /kaggle/input:

- GPU 0: Qwen3-VL-4B-Instruct cập nhật ngữ nghĩa nền khoảng 1 Hz.
- GPU 1: Depth Anything V2 chạy mỗi frame, mục tiêu 5 FPS; PhoWhisper chạy khi nhấn nói.
- CPU: VieNeu-TTS.
- Gemini Flash: mô tả cảnh qua endpoint /v1/describe để phản hồi nhanh.
- FastAPI chạy ngay trong kernel, không tạo file server Python.
- Cloudflare Quick Tunnel cung cấp HTTPS/WSS cho điện thoại, có retry khi DNS chưa cập nhật.

Trước khi bấm Run All:

1. Tạo Kaggle Dataset private từ mã nguồn dự án rồi Add Input vào notebook; Kaggle sẽ tự giải nén.
2. Chọn Accelerator = GPU T4 x2 và bật Internet.
3. Thêm Kaggle Secrets HF_TOKEN (khuyên dùng) và GEMINI_API_KEY (bắt buộc cho Describe).
4. Nếu session từng chạy bản cũ và lỗi Pillow/NumPy, chọn Factory Reset (hoặc Restart Session) rồi Run All.
5. Giữ cell cuối chạy để phiên Kaggle và tunnel tiếp tục hoạt động.

Đây là demo hỗ trợ thử nghiệm, chưa phải thiết bị dẫn đường an toàn đã được chứng nhận.


In [ ]:
# 1. Cài dependency mà không thay ABI NumPy/SciPy/Pillow của Kaggle
from importlib.metadata import version as dist_version
from pathlib import Path
import subprocess
import sys

ABI_PACKAGES = ("numpy", "scipy", "Pillow")
ABI_VERSIONS_BEFORE = {name: dist_version(name) for name in ABI_PACKAGES}
ABI_CONSTRAINTS = Path("/tmp/kaggle-abi-constraints.txt")
ABI_CONSTRAINTS.write_text(
    "".join(f"{name}=={version}\n" for name, version in ABI_VERSIONS_BEFORE.items()),
    encoding="utf-8",
)
print("Giữ nguyên Kaggle ABI:", ABI_VERSIONS_BEFORE)

INSTALL_PACKAGES = [
    "transformers==4.57.6",
    "accelerate>=1.0.0",
    "safetensors",
    "huggingface_hub>=0.34.0,<1.0",
    "fastapi>=0.115.0",
    "uvicorn[standard]>=0.30.0",
    "websockets>=12.0",
    "python-multipart",
    "librosa>=0.11.0",
    "soundfile>=0.12.0",
    "vieneu==3.6.4",
    "requests>=2.31.0",
    "google-genai",
]
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--constraint",
        str(ABI_CONSTRAINTS),
        *INSTALL_PACKAGES,
    ],
    check=True,
)

ABI_VERSIONS_AFTER = {name: dist_version(name) for name in ABI_PACKAGES}
assert ABI_VERSIONS_AFTER == ABI_VERSIONS_BEFORE, (
    "pip đã đổi thư viện ABI; hãy Factory Reset session. "
    f"before={ABI_VERSIONS_BEFORE}, after={ABI_VERSIONS_AFTER}"
)
try:
    import numpy as _numpy_check
    import scipy as _scipy_check
    from PIL import Image as _pillow_check
except ImportError as exc:
    raise RuntimeError(
        "Kernel đang giữ module cũ sau lần pip trước. Chọn Restart Session "
        "hoặc Factory Reset, sau đó Run All từ đầu."
    ) from exc
print(
    "ABI import OK:",
    _numpy_check.__version__,
    _scipy_check.__version__,
    dist_version("Pillow"),
)


In [ ]:
# 2. Chuẩn bị runtime, GPU, mã nguồn private và secrets
import base64
import io
import os
import re
import secrets
import shutil
import subprocess
import sys
import tempfile
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests
import torch

assert torch.cuda.is_available(), "Kaggle chưa bật GPU."
assert torch.cuda.device_count() >= 2, (
    f"Notebook cần GPU T4 x2; hiện chỉ thấy {torch.cuda.device_count()} GPU."
)
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(2)]
print("GPU:", GPU_NAMES)

WORKDIR = Path("/kaggle/working/ai-assistant")
INPUT_ROOT = Path("/kaggle/input")

# Kaggle tự giải nén mỗi Dataset/Input. Tìm đúng thư mục gốc của dự án
# bằng các file đặc trưng, kể cả khi Dataset có thêm một thư mục bọc bên ngoài.
source_roots = []
for marker in INPUT_ROOT.glob("**/scripts/mvp_config.py"):
    candidate = marker.parent.parent
    if all((
        (candidate / "web" / "index.html").is_file(),
        (candidate / "scripts" / "fusion.py").is_file(),
        (candidate / "scripts" / "depth_path.py").is_file(),
        (candidate / "assets" / "audio").is_dir(),
    )):
        source_roots.append(candidate.resolve())

source_roots = sorted(set(source_roots))
assert source_roots, (
    "Không tìm thấy mã nguồn đã giải nén trong /kaggle/input. "
    "Hãy Add Input chứa web/, scripts/ và assets/ vào notebook."
)
assert len(source_roots) == 1, (
    "Tìm thấy nhiều bản mã nguồn; chỉ attach một Dataset dự án: "
    f"{source_roots}"
)
SOURCE_ROOT = source_roots[0]

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
shutil.copytree(SOURCE_ROOT, WORKDIR)

required = [
    WORKDIR / "web" / "index.html",
    WORKDIR / "scripts" / "mvp_config.py",
    WORKDIR / "scripts" / "fusion.py",
    WORKDIR / "scripts" / "depth_path.py",
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f"Gói mã nguồn thiếu file bắt buộc: {missing}"

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR / "scripts"))
os.environ["HF_HOME"] = str(WORKDIR / "models" / "hf")
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    from kaggle_secrets import UserSecretsClient
    secrets_client = UserSecretsClient()
except Exception:
    secrets_client = None

try:
    hf_token = secrets_client.get_secret("HF_TOKEN") if secrets_client else None
except Exception:
    hf_token = None

try:
    gemini_api_key = secrets_client.get_secret("GEMINI_API_KEY") if secrets_client else None
except Exception:
    gemini_api_key = None

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN: đã nạp")
else:
    print("HF_TOKEN: không có; model public vẫn tải được")

if gemini_api_key:
    os.environ["GEMINI_API_KEY"] = gemini_api_key
    print("GEMINI_API_KEY: đã nạp")
else:
    print("GEMINI_API_KEY: chưa có; /v1/describe sẽ trả HTTP 503")

import mvp_config as cfg
import fusion as fus

cfg.MODELS_DIR = str(WORKDIR / "models")
cfg.ALERT_AUDIO_DIR = str(WORKDIR / "assets" / "audio")

audio_dir = WORKDIR / "assets" / "audio"
missing_audio = [code for code in cfg.ALERT_PHRASES if not (audio_dir / f"{code}.wav").exists()]
assert not missing_audio, f"Thiếu WAV cache: {missing_audio}"
print("Source input:", SOURCE_ROOT)
print("Runtime:", WORKDIR)
print("Models:", cfg.VLM_MODEL, cfg.DEPTH_MODEL, cfg.STT_MODEL, cfg.TTS_MODEL)


In [ ]:
# Chọn model Gemini dùng cho endpoint /v1/describe.
os.environ["DESCRIBE_MODEL"] = "gemini-3.5-flash-lite"
print("Describe model:", os.environ["DESCRIBE_MODEL"])

In [ ]:
# 3. Tải và nạp toàn bộ model trực tiếp trong kernel
from transformers import (
    AutoImageProcessor,
    AutoModelForDepthEstimation,
    AutoModelForImageTextToText,
    AutoProcessor,
)
from transformers import WhisperForConditionalGeneration
from vieneu import Vieneu

MODEL_STATE = {}
VISION_EXECUTOR = ThreadPoolExecutor(max_workers=2, thread_name_prefix="vision")
VLM_LOCK = threading.Lock()
GPU1_LOCK = threading.Lock()
TTS_LOCK = threading.Lock()

def load_models():
    print("1/4 Loading Qwen3-VL-4B on cuda:0 ...")
    MODEL_STATE["vlm_processor"] = AutoProcessor.from_pretrained(
        cfg.VLM_MODEL, token=hf_token
    )
    visual_scale = 32 * 32
    MODEL_STATE["vlm_processor"].image_processor.size = {
        "shortest_edge": getattr(cfg, "VLM_MIN_VISUAL_TOKENS", 128) * visual_scale,
        "longest_edge": getattr(cfg, "VLM_MAX_VISUAL_TOKENS", 320) * visual_scale,
    }
    print(
        "Qwen visual-token budget:",
        getattr(cfg, "VLM_MIN_VISUAL_TOKENS", 128),
        "..",
        getattr(cfg, "VLM_MAX_VISUAL_TOKENS", 320),
    )
    MODEL_STATE["vlm"] = AutoModelForImageTextToText.from_pretrained(
        cfg.VLM_MODEL,
        token=hf_token,
        dtype=torch.float16,
        attn_implementation=cfg.ATTN_IMPL,
        device_map="cuda:0",
        low_cpu_mem_usage=True,
    ).eval()

    print("2/4 Loading Depth Anything V2 Indoor Large on cuda:1 ...")
    MODEL_STATE["depth_processor"] = AutoImageProcessor.from_pretrained(
        cfg.DEPTH_MODEL, token=hf_token
    )
    MODEL_STATE["depth"] = AutoModelForDepthEstimation.from_pretrained(
        cfg.DEPTH_MODEL,
        token=hf_token,
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    ).to("cuda:1").eval()

    print("3/4 Loading PhoWhisper-medium on cuda:1 ...")
    MODEL_STATE["stt_processor"] = AutoProcessor.from_pretrained(
        cfg.STT_MODEL, token=hf_token
    )
    MODEL_STATE["stt"] = WhisperForConditionalGeneration.from_pretrained(
        cfg.STT_MODEL,
        token=hf_token,
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    ).to("cuda:1").eval()

    print("4/4 Loading VieNeu-TTS on CPU ...")
    MODEL_STATE["tts"] = Vieneu()
    MODEL_STATE["ready"] = True
    print("Tất cả model đã sẵn sàng.")

load_models()

for device_id in range(2):
    allocated = torch.cuda.memory_allocated(device_id) / 1024**3
    reserved = torch.cuda.memory_reserved(device_id) / 1024**3
    print(f"cuda:{device_id}: allocated={allocated:.2f} GiB, reserved={reserved:.2f} GiB")


In [ ]:
# 4. Inference: VLM và depth chạy song song trên hai GPU
import numpy as np
from PIL import Image

MAX_IMAGE_BYTES = 5 * 1024 * 1024
MAX_AUDIO_BYTES = 10 * 1024 * 1024
STT_CONFIDENCE_THRESHOLD = 0.60

def decode_base64(value: str, limit: int) -> bytes:
    if not isinstance(value, str) or not value:
        raise ValueError("base64 payload is required")
    if "," in value and value.lstrip().lower().startswith("data:"):
        value = value.split(",", 1)[1]
    if len(value) > ((limit + 2) // 3) * 4 + 16:
        raise ValueError("payload exceeds size limit")
    try:
        data = base64.b64decode(value, validate=True)
    except Exception as exc:
        raise ValueError("invalid base64 payload") from exc
    if len(data) > limit:
        raise ValueError("payload exceeds size limit")
    return data

def prepare_image(data: bytes) -> Image.Image:
    if len(data) > MAX_IMAGE_BYTES:
        raise ValueError("image exceeds 5 MiB")
    image = Image.open(io.BytesIO(data))
    image.load()
    image = image.convert("RGB")
    if max(image.size) > cfg.FRAME_LONG_EDGE:
        scale = cfg.FRAME_LONG_EDGE / max(image.size)
        image = image.resize(
            (round(image.width * scale), round(image.height * scale)),
            Image.Resampling.LANCZOS,
        )
    return image

def run_vlm(image: Image.Image, system_prompt: str, user_text: str, max_new_tokens: int):
    processor = MODEL_STATE["vlm_processor"]
    model = MODEL_STATE["vlm"]
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": user_text},
        ]},
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda:0")
    prompt_tokens = inputs["input_ids"].shape[-1]
    with VLM_LOCK, torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )
    return processor.decode(
        output[0][prompt_tokens:], skip_special_tokens=True
    ).strip()

def run_hazard_vlm(image: Image.Image):
    raw = run_vlm(image, cfg.VLM_PROMPT_PIPE, "Frame:", cfg.VLM_MAX_NEW_TOKENS)
    return raw, cfg.parse_pipe4(raw)

def run_depth(image: Image.Image):
    processor = MODEL_STATE["depth_processor"]
    model = MODEL_STATE["depth"]
    with GPU1_LOCK:
        inputs = processor(images=image, return_tensors="pt").to("cuda:1")
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)
        with torch.inference_mode():
            output = model(**inputs)
        depth = processor.post_process_depth_estimation(
            output, target_sizes=[(image.height, image.width)]
        )[0]["predicted_depth"]
        return depth.float().cpu().numpy().squeeze()

def _timed_hazard(image: Image.Image):
    started = time.perf_counter()
    raw, parsed = run_hazard_vlm(image)
    return raw, parsed, round((time.perf_counter() - started) * 1000)

def _depth_result(image: Image.Image):
    started = time.perf_counter()
    depth = run_depth(image)
    return depth, round((time.perf_counter() - started) * 1000)

def _vision_payload(started, raw, parsed, depth, depth_ms, vlm_ms, fresh, age_ms):
    return {
        "latency_ms": round((time.perf_counter() - started) * 1000),
        "pipeline_mode": "depth_realtime_vlm_cached",
        "vlm_fresh": fresh,
        "vlm_age_ms": age_ms,
        "vlm_latency_ms": vlm_ms,
        "depth_latency_ms": depth_ms,
        "vlm_raw": raw,
        "vlm": parsed,
        "depth": {
            "min": float(np.min(depth)),
            "median": float(np.median(depth)),
            "max": float(np.max(depth)),
        },
        "_depth_map": depth,
    }

def infer_vision(image: Image.Image):
    """Đường đồng bộ dùng cho warm-up và benchmark một frame."""
    started = time.perf_counter()
    vlm_future = VISION_EXECUTOR.submit(_timed_hazard, image.copy())
    depth, depth_ms = _depth_result(image)
    raw, parsed, vlm_ms = vlm_future.result()
    return _vision_payload(
        started, raw, parsed, depth, depth_ms, vlm_ms, True, 0
    )

def new_vision_state():
    return {
        "vlm_future": None,
        "vlm_raw": None,
        "vlm": None,
        "vlm_latency_ms": None,
        "vlm_completed_at": 0.0,
        "vlm_submitted_at": 0.0,
        "vlm_error": None,
    }

def _collect_vlm(state, *, wait=False):
    future = state["vlm_future"]
    if future is None or (not wait and not future.done()):
        return False
    try:
        raw, parsed, latency_ms = future.result()
        state.update({
            "vlm_raw": raw,
            "vlm": parsed,
            "vlm_latency_ms": latency_ms,
            "vlm_completed_at": time.perf_counter(),
            "vlm_error": None,
        })
        return True
    except Exception as exc:
        state["vlm_error"] = str(exc)
        if state["vlm_raw"] is None:
            raise
        return False
    finally:
        state["vlm_future"] = None

def infer_vision_stream(image: Image.Image, state):
    """Depth mỗi frame; Qwen chạy nền và dùng cache ngắn hạn giữa các frame."""
    started = time.perf_counter()
    fresh = _collect_vlm(state)
    now = time.perf_counter()
    refresh_hz = max(0.1, getattr(cfg, "VLM_REFRESH_HZ", 1.0))
    refresh_interval = 1.0 / refresh_hz

    if (
        state["vlm_future"] is None
        and (
            state["vlm_raw"] is None
            or now - state["vlm_submitted_at"] >= refresh_interval
        )
    ):
        state["vlm_future"] = VISION_EXECUTOR.submit(
            _timed_hazard, image.copy()
        )
        state["vlm_submitted_at"] = now

    depth, depth_ms = _depth_result(image)
    fresh = _collect_vlm(state) or fresh

    # Frame đầu tiên phải có đủ semantics + depth; từ frame sau không chặn depth.
    if state["vlm_raw"] is None:
        fresh = _collect_vlm(state, wait=True) or fresh

    age_ms = round(
        max(0.0, time.perf_counter() - state["vlm_completed_at"]) * 1000
    )
    max_stale_ms = round(
        getattr(cfg, "VLM_MAX_STALE_S", 2.5) * 1000
    )
    parsed = state["vlm"] if age_ms <= max_stale_ms else None

    return _vision_payload(
        started,
        state["vlm_raw"],
        parsed,
        depth,
        depth_ms,
        state["vlm_latency_ms"],
        fresh,
        age_ms,
    )

DESCRIBE_MODEL = os.getenv("DESCRIBE_MODEL", "gemini-3.5-flash-lite")
_GEMINI_CLIENT = None
_GEMINI_LOCK = threading.Lock()

def extract_final_text(response) -> str:
    """Read structured answer, falling back to non-thought text parts."""
    parsed = getattr(response, "parsed", None)
    if isinstance(parsed, dict) and isinstance(parsed.get("answer"), str):
        return parsed["answer"].strip()
    candidates = getattr(response, "candidates", None) or []
    content = getattr(candidates[0], "content", None) if candidates else None
    parts = getattr(content, "parts", None) or []
    answer = [
        str(part.text).strip()
        for part in parts
        if getattr(part, "text", None)
        and not getattr(part, "thought", False)
    ]
    raw = "\n".join(filter(None, answer)).strip()
    try:
        payload = json.loads(raw)
        if isinstance(payload, dict) and isinstance(payload.get("answer"), str):
            return payload["answer"].strip()
    except (TypeError, ValueError):
        pass
    return raw

def description_is_incomplete(response, text: str) -> bool:
    """Reject token-cut or sentence-fragment Gemini answers."""
    candidates = getattr(response, "candidates", None) or []
    reason = getattr(candidates[0], "finish_reason", "") if candidates else ""
    reason_name = getattr(reason, "name", str(reason)).upper()
    clean = text.strip()
    tail = clean.rstrip(".!?…").casefold().split()
    dangling = {
        "đang", "sẽ", "và", "hoặc", "nhưng", "ở", "có", "là", "một",
        "những", "các", "phía", "bên", "trên", "dưới", "trong", "với",
        "để", "của", "khi", "do", "từ", "đến", "về",
    }
    return (
        "MAX_TOKENS" in reason_name
        or len(clean.split()) < 4
        or clean[-1:] not in ".!?…"
        or bool(tail and tail[-1] in dangling)
    )

def describe_with_gemini(image: Image.Image, question: str):
    global _GEMINI_CLIENT
    if not os.getenv("GEMINI_API_KEY"):
        raise RuntimeError("GEMINI_API_KEY is not configured")
    from google import genai
    from google.genai import types
    with _GEMINI_LOCK:
        if _GEMINI_CLIENT is None:
            _GEMINI_CLIENT = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
        prompt = (
            "Bạn là trợ lý thị giác cho người khiếm thị. Trả lời tối đa hai câu "
            "tiếng Việt ngắn, đúng sự thật. Nêu vật hoặc người quan trọng nhất, "
            "vị trí trái/phải/phía trước và điều cần chú ý ngay. Không suy đoán "
            "chi tiết không có trong ảnh. Luôn viết câu hoàn chỉnh và kết thúc "
            "bằng dấu câu. Không kết thúc câu bằng từ nối hoặc động từ phụ như "
            "\"đang\", \"sẽ\", \"và\". Yêu cầu: " + question.strip()[:500]
        )
        config = types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),
            response_mime_type="application/json",
            response_schema={
                "type": "OBJECT",
                "required": ["answer"],
                "properties": {
                    "answer": {
                        "type": "STRING",
                        "description": "Một hoặc hai câu tiếng Việt hoàn chỉnh.",
                    },
                },
            },
        )
        response = _GEMINI_CLIENT.models.generate_content(
            model=DESCRIBE_MODEL,
            contents=[prompt, image],
            config=config,
        )
    text = extract_final_text(response)
    if description_is_incomplete(response, text):
        raise RuntimeError(
            f"Gemini returned an incomplete response: {text!r}"
        )
    return text

def transcribe(speech, sample_rate: int):
    processor = MODEL_STATE["stt_processor"]
    model = MODEL_STATE["stt"]
    with GPU1_LOCK:
        features = processor(
            speech, sampling_rate=sample_rate, return_tensors="pt"
        ).input_features.to("cuda:1", dtype=torch.float16)
        with torch.inference_mode():
            generated = model.generate(
                features,
                language="vi",
                task="transcribe",
                max_new_tokens=96,
                do_sample=False,
                num_beams=1,
                return_dict_in_generate=True,
                output_scores=True,
            )
    text = processor.batch_decode(
        generated.sequences, skip_special_tokens=True
    )[0].strip()
    if not generated.scores:
        return text, 0.0
    generated_ids = generated.sequences[0, -len(generated.scores):].cpu()
    log_probs = []
    for score, token_id in zip(generated.scores, generated_ids):
        token_log_prob = torch.log_softmax(score[0].float().cpu(), dim=-1)[token_id]
        log_probs.append(token_log_prob)
    confidence = float(torch.stack(log_probs).mean().exp()) if log_probs else 0.0
    return text, confidence

def synthesize_wav(text: str) -> bytes:
    with TTS_LOCK:
        audio = MODEL_STATE["tts"].infer(text, voice_id=cfg.TTS_VOICE)
        with tempfile.NamedTemporaryFile(suffix=".wav") as output:
            MODEL_STATE["tts"].save(audio, output.name)
            return Path(output.name).read_bytes()

# Tạo WAV hệ thống một lần khi khởi động, dùng lại cho mọi lượt hội thoại.
for code, text in cfg.SYSTEM_PHRASES.items():
    cached = WORKDIR / "assets" / "audio" / f"{code}.wav"
    if not cached.is_file():
        cached.write_bytes(synthesize_wav(text))
print("Voice prompts cached:", len(cfg.SYSTEM_PHRASES))

# Warm-up thật trên một frame của repository.
sample_path = WORKDIR / "assets" / "frames" / "clearpath_00.jpg"
with Image.open(sample_path) as sample:
    warmup = infer_vision(prepare_image(sample_path.read_bytes()))
depth_map = warmup.pop("_depth_map")
warmup["geometry"] = {
    key: value
    for key, value in fus.depth_evidence(depth_map).items()
    if key != "per_col_rank"
}
print("Warm-up:", warmup)


In [ ]:
# 5. FastAPI khai báo trực tiếp trong notebook, không sinh file .py
import asyncio
from fastapi import FastAPI, File, Form, HTTPException, Request, UploadFile, WebSocket
from fastapi import WebSocketDisconnect
from fastapi.responses import FileResponse, JSONResponse
from starlette.staticfiles import StaticFiles

ACCESS_TOKEN = secrets.token_urlsafe(24)
COOKIE_NAME = "ai_assistant_session"
app = FastAPI(title="Self-hosted Indoor AI Assistant", version="1.0")

def with_cached_reply_audio(route: dict) -> dict:
    result = dict(route)
    code = result.get("reply_code")
    path = audio_dir / f"{code}.wav" if code else None
    result["reply_audio_url"] = (
        f"/audio/{code}.wav" if path is not None and path.is_file() else None
    )
    return result

def token_matches(value):
    return isinstance(value, str) and secrets.compare_digest(value, ACCESS_TOKEN)

@app.middleware("http")
async def protect_public_routes(request: Request, call_next):
    if request.url.path == "/health":
        return await call_next(request)
    supplied = request.query_params.get("token")
    authorized = token_matches(supplied) or token_matches(request.cookies.get(COOKIE_NAME))
    if not authorized:
        return JSONResponse({"error": "unauthorized"}, status_code=401)
    response = await call_next(request)
    if token_matches(supplied):
        response.set_cookie(
            COOKIE_NAME,
            ACCESS_TOKEN,
            secure=True,
            httponly=True,
            samesite="lax",
            max_age=12 * 60 * 60,
        )
    return response

@app.get("/health")
async def health():
    return {
        "status": "ok" if MODEL_STATE.get("ready") else "loading",
        "self_hosted": True,
        "models": {
            "vlm": cfg.VLM_MODEL,
            "depth": cfg.DEPTH_MODEL,
            "stt": cfg.STT_MODEL,
            "tts": cfg.TTS_MODEL,
            "describe": DESCRIBE_MODEL,
        },
        "gpus": {
            "cuda:0": "VLM",
            "cuda:1": "Depth + STT",
        },
    }

@app.get("/")
async def index():
    page = WORKDIR / "web" / "dist" / "index.html"
    return FileResponse(page if page.is_file() else WORKDIR / "web" / "index.html",
                        headers={"Cache-Control": "no-store"})

ui_assets = WORKDIR / "web" / "dist" / "ui-assets"
if ui_assets.is_dir():
    app.mount("/ui-assets", StaticFiles(directory=str(ui_assets)), name="ui-assets")

@app.get("/v1/alerts")
async def alerts_manifest():
    return {
        "phrases": cfg.ALERT_PHRASES,
        "audio_url_pattern": "/audio/{code}.wav",
        "audio_kind": "emergency",
        "audio_priority": cfg.VOICE_AUDIO_PRIORITIES["emergency"],
    }

@app.get("/v1/bootstrap")
async def bootstrap():
    return {
        "default_mode": "waiting",
        "welcome": {
            "reply_code": "WELCOME",
            "reply_text": cfg.SYSTEM_PHRASES["WELCOME"],
            "audio_kind": cfg.SYSTEM_PHRASE_KINDS["WELCOME"],
            "audio_priority": cfg.VOICE_AUDIO_PRIORITIES["confirmation"],
        },
        "system_phrases": cfg.SYSTEM_PHRASES,
        "system_audio": {
            code: (
                f"/audio/{code}.wav"
                if (audio_dir / f"{code}.wav").is_file()
                else None
            )
            for code in cfg.SYSTEM_PHRASES
        },
        "audio_priorities": cfg.VOICE_AUDIO_PRIORITIES,
        "voice_capture": {
            "silence_timeout_ms": 800,
            "max_utterance_ms": 8000,
            "resume_after_playback_ms": 300,
        },
    }

@app.post("/v1/intent")
async def intent_endpoint(payload: dict):
    text = str(payload.get("text", "")).strip()
    if not text:
        raise HTTPException(400, "text is required")
    return with_cached_reply_audio(
        cfg.route_voice_intent(
            text,
            str(payload.get("current_mode", "waiting")),
            str(payload.get("resume_mode", "guide")),
        )
    )

app.mount("/audio", StaticFiles(directory=str(audio_dir)), name="audio")

@app.post("/v1/stt")
async def stt_endpoint(
    audio: UploadFile = File(...),
    current_mode: str = Form("waiting"),
    resume_mode: str = Form("guide"),
    turn_id: str = Form(""),
):
    data = await audio.read(MAX_AUDIO_BYTES + 1)
    if len(data) > MAX_AUDIO_BYTES:
        raise HTTPException(413, "audio exceeds 10 MiB")
    try:
        import librosa
        speech, sample_rate = librosa.load(
            io.BytesIO(data), sr=16000, mono=True
        )
        text, confidence = await asyncio.to_thread(
            transcribe, speech, sample_rate
        )
    except Exception as exc:
        raise HTTPException(422, f"invalid audio: {exc}") from exc

    result = {
        "turn_id": turn_id[:128] or None,
        "text": text,
        "confidence": round(confidence, 4),
        "repeat": confidence < STT_CONFIDENCE_THRESHOLD,
    }
    if result["repeat"]:
        result.update({
            "intent": None,
            "mode": current_mode,
            "reply_code": "REPEAT_PLEASE",
            "audio_kind": "repeat",
            "audio_priority": cfg.VOICE_AUDIO_PRIORITIES["repeat"],
            "reply_audio_url": "/audio/REPEAT_PLEASE.wav",
        })
        repeat_path = audio_dir / "REPEAT_PLEASE.wav"
        if repeat_path.exists():
            result["audio_base64"] = base64.b64encode(
                repeat_path.read_bytes()
            ).decode()
    else:
        result.update(with_cached_reply_audio(
            cfg.route_voice_intent(text, current_mode, resume_mode)
        ))
    return result

@app.post("/v1/tts")
async def tts_endpoint(payload: dict):
    text = str(payload.get("text", "")).strip()
    if not 1 <= len(text) <= 500:
        raise HTTPException(400, "text must be 1-500 characters")

    code = next(
        (key for key, phrase in {**cfg.ALERT_PHRASES, **cfg.SYSTEM_PHRASES}.items() if phrase == text),
        None,
    )
    if code:
        cached_path = audio_dir / f"{code}.wav"
        if cached_path.exists():
            return {
                "message_code": code,
                "cached": True,
                "audio_base64": base64.b64encode(
                    cached_path.read_bytes()
                ).decode(),
            }

    wav = await asyncio.to_thread(synthesize_wav, text)
    return {
        "message_code": None,
        "cached": False,
        "audio_base64": base64.b64encode(wav).decode(),
    }

@app.post("/v1/describe")
async def describe_endpoint(payload: dict):
    raw = payload.get("image_base64")
    question = str(payload.get("question", "Mô tả cảnh trước mặt."))
    want_audio = bool(payload.get("tts", True))
    request_id = str(payload.get("request_id", "")).strip()[:128] or None
    if not os.getenv("GEMINI_API_KEY"):
        raise HTTPException(503, "GEMINI_API_KEY is not configured")
    try:
        image = prepare_image(decode_base64(raw, MAX_IMAGE_BYTES))
        text = await asyncio.to_thread(describe_with_gemini, image, question)
        result = {
            "request_id": request_id,
            "model": DESCRIBE_MODEL,
            "provider": "gemini",
            "text": text,
            "audio_kind": "answer",
            "audio_priority": cfg.VOICE_AUDIO_PRIORITIES["answer"],
            "interruptible": True,
        }
        if want_audio:
            wav = await asyncio.to_thread(synthesize_wav, text)
            result["audio_base64"] = base64.b64encode(wav).decode()
        return result
    except ValueError as exc:
        raise HTTPException(400, str(exc)) from exc
    except Exception as exc:
        raise HTTPException(500, f"Gemini description failed: {exc}") from exc

@app.websocket("/v1/vision")
async def vision_endpoint(ws: WebSocket):
    authorized = token_matches(ws.query_params.get("token")) or token_matches(
        ws.cookies.get(COOKIE_NAME)
    )
    if not authorized:
        await ws.close(code=1008, reason="unauthorized")
        return

    await ws.accept()
    session = fus.FusionSession()
    vision_state = new_vision_state()
    latest = asyncio.Queue(maxsize=1)
    dropped_frames = 0

    def process_frame(message, received_at, dropped):
        queue_wait_ms = round((time.perf_counter() - received_at) * 1000)
        image = prepare_image(
            decode_base64(message.get("image_base64"), MAX_IMAGE_BYTES)
        )
        result = infer_vision_stream(image, vision_state)
        depth_map = result.pop("_depth_map")
        vlm_evidence = fus.vlm_evidence(result.get("vlm"))
        geometry = fus.depth_evidence(depth_map)
        decision = fus.decide_frame(
            vlm_evidence, geometry, session.config
        )
        event = fus.update_session(
            session,
            decision,
            vlm_evidence,
            new_semantics=result.get("vlm_fresh", False),
        )
        return {
            "ok": True,
            **result,
            "frame_id": message.get("frame_id"),
            "dropped_frames": dropped,
            "queue_wait_ms": queue_wait_ms,
            "geometry": {
                key: value
                for key, value in geometry.items()
                if key != "per_col_rank"
            },
            "decision": event,
        }

    async def receive_latest():
        nonlocal dropped_frames
        try:
            while True:
                item = (await ws.receive_json(), time.perf_counter())
                if latest.full():
                    latest.get_nowait()
                    dropped_frames += 1
                latest.put_nowait((*item, dropped_frames))
        except WebSocketDisconnect:
            return

    async def infer_latest():
        while True:
            message, received_at, dropped = await latest.get()
            try:
                payload = await asyncio.to_thread(
                    process_frame, message, received_at, dropped
                )
            except Exception as exc:
                payload = {
                    "ok": False,
                    "error": str(exc),
                    "frame_id": message.get("frame_id"),
                    "dropped_frames": dropped,
                }
            payload["server_total_ms"] = round(
                (time.perf_counter() - received_at) * 1000
            )
            await ws.send_json(payload)

    try:
        receiver = asyncio.create_task(receive_latest())
        worker = asyncio.create_task(infer_latest())
        done, pending = await asyncio.wait(
            {receiver, worker}, return_when=asyncio.FIRST_COMPLETED
        )
        for task in pending:
            task.cancel()
        await asyncio.gather(*pending, return_exceptions=True)
        for task in done:
            task.result()
    except WebSocketDisconnect:
        pass


In [ ]:
# 6. Khởi động server trong kernel và mở Cloudflare Quick Tunnel
import hashlib
import uvicorn

def get_with_retry(
    session,
    url,
    *,
    attempts,
    timeout,
    label,
    base_delay=2,
    max_delay=10,
):
    """GET có retry cho DNS, timeout, HTTP 429 và lỗi máy chủ."""
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            response = session.get(url, timeout=timeout)
            if response.status_code == 429 or response.status_code >= 500:
                response.raise_for_status()
            response.raise_for_status()
            return response
        except requests.RequestException as exc:
            last_error = exc
            if attempt == attempts:
                break
            delay = min(base_delay * (2 ** (attempt - 1)), max_delay)
            print(
                f"{label}: lần {attempt}/{attempts} chưa thành công "
                f"({type(exc).__name__}); thử lại sau {delay}s..."
            )
            time.sleep(delay)
    raise RuntimeError(
        f"{label} thất bại sau {attempts} lần: {last_error}"
    ) from last_error

old_server = globals().get("uvicorn_server")
old_thread = globals().get("uvicorn_thread")
if old_server is not None:
    old_server.should_exit = True
if old_thread is not None and old_thread.is_alive():
    old_thread.join(timeout=10)

uvicorn_server = uvicorn.Server(
    uvicorn.Config(
        app,
        host="127.0.0.1",
        port=8000,
        log_level="warning",
        access_log=False,
    )
)
uvicorn_thread = threading.Thread(
    target=uvicorn_server.run,
    daemon=True,
    name="uvicorn",
)
uvicorn_thread.start()

for _ in range(60):
    try:
        response = requests.get("http://127.0.0.1:8000/health", timeout=1)
        if response.ok and response.json().get("status") == "ok":
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("FastAPI không sẵn sàng sau 60 giây")
print("Local API:", response.json())

CLOUDFLARED_VERSION = "2026.7.2"
CLOUDFLARED_SHA256 = "ec905ea7b7e327ff8abdde8cb64697a2152de74dbcdbf6aec9db8364eb3886cd"
CLOUDFLARED = Path("/kaggle/working/cloudflared")
download_url = (
    "https://github.com/cloudflare/cloudflared/releases/download/"
    f"{CLOUDFLARED_VERSION}/cloudflared-linux-amd64"
)

if (
    not CLOUDFLARED.exists()
    or hashlib.sha256(CLOUDFLARED.read_bytes()).hexdigest()
    != CLOUDFLARED_SHA256
):
    download = get_with_retry(
        requests.Session(),
        download_url,
        attempts=5,
        timeout=(15, 180),
        label="Tải cloudflared",
    )
    digest = hashlib.sha256(download.content).hexdigest()
    assert digest == CLOUDFLARED_SHA256, (
        f"cloudflared checksum mismatch: {digest}"
    )
    CLOUDFLARED.write_bytes(download.content)
    CLOUDFLARED.chmod(0o755)

old_tunnel = globals().get("tunnel_proc")
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=10)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)

tunnel_log_path = Path("/kaggle/working/cloudflared.log")
tunnel_log = open(tunnel_log_path, "w", buffering=1)
tunnel_proc = subprocess.Popen(
    [
        str(CLOUDFLARED),
        "tunnel",
        "--url",
        "http://127.0.0.1:8000",
        "--no-autoupdate",
        "--edge-ip-version",
        "4",
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    text=True,
)

PUBLIC_URL = None
deadline = time.time() + 90
while time.time() < deadline:
    log_text = (
        tunnel_log_path.read_text(errors="replace")
        if tunnel_log_path.exists()
        else ""
    )
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        PUBLIC_URL = match.group(0)
        break
    if tunnel_proc.poll() is not None:
        break
    time.sleep(1)

assert PUBLIC_URL, (
    "Không tạo được Cloudflare tunnel:\n"
    + tunnel_log_path.read_text(errors="replace")[-5000:]
)

PHONE_URL = f"{PUBLIC_URL}/?token={ACCESS_TOKEN}"
public_session = requests.Session()

# Quick Tunnel có thể ghi URL trước khi DNS toàn cầu sẵn sàng.
# Chờ tối đa khoảng hai phút thay vì làm hỏng cả cell ngay lần đầu.
root_probe = get_with_retry(
    public_session,
    PHONE_URL,
    attempts=12,
    timeout=(10, 30),
    label="Chờ Cloudflare DNS/public URL",
)
alerts_probe = get_with_retry(
    public_session,
    PUBLIC_URL + "/v1/alerts",
    attempts=6,
    timeout=(10, 30),
    label="Kiểm tra alerts endpoint",
)

print("=" * 72)
print("HỆ THỐNG ĐÃ SẴN SÀNG")
print("Mở URL này trên điện thoại:")
print(PHONE_URL)
print("API health:", PUBLIC_URL + "/health")
print("Cảnh báo/STT/TTS chạy local; Describe dùng Gemini Flash.")
print("Không tạo server .py phụ.")
print("=" * 72)


In [ ]:
# 7. Giữ phiên sống và hiển thị trạng thái; Stop cell để kết thúc
from IPython.display import clear_output

watch_started = time.time()
try:
    while True:
        if not uvicorn_thread.is_alive():
            raise RuntimeError("Uvicorn đã dừng")
        if tunnel_proc.poll() is not None:
            tail = tunnel_log_path.read_text(errors="replace")[-3000:]
            raise RuntimeError(f"Cloudflare tunnel đã dừng:\n{tail}")

        clear_output(wait=True)
        elapsed = int(time.time() - watch_started)
        print(f"RUNNING {elapsed // 60}m {elapsed % 60:02d}s")
        print("Điện thoại:", PHONE_URL)
        for device_id, role in ((0, "Qwen3-VL"), (1, "Depth + PhoWhisper")):
            allocated = torch.cuda.memory_allocated(device_id) / 1024**3
            reserved = torch.cuda.memory_reserved(device_id) / 1024**3
            print(
                f"cuda:{device_id} {role}: "
                f"{allocated:.2f} GiB allocated, {reserved:.2f} GiB reserved"
            )
        print("Nhấn Stop cell khi muốn dừng.")
        time.sleep(10)
except KeyboardInterrupt:
    print("Đã dừng watcher; server sẽ sống tới khi kernel Kaggle kết thúc.")
